# Spectral Graph Theory — Exercises

8 graded exercises covering Laplacian construction, spectrum of special graphs, Fiedler vector, Cheeger inequality, Graph Fourier Transform, spectral clustering, Laplacian positional encodings, and PageRank.

| Format | Description |
|--------|-------------|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding (`# YOUR CODE HERE`) |
| **Solution** | Reference solution with PASS/FAIL checks |

### Difficulty Levels

| Level | Exercises | Focus |
|-------|-----------|-------|
| ★ | 1-3 | Core mechanics |
| ★★ | 4-6 | Deeper theory |
| ★★★ | 7-8 | AI / ML applications |

### Topic Map

| Topic | Exercise |
|-------|----------|
| Laplacian construction and Dirichlet energy | 1 |
| Spectrum of special graphs | 2 |
| Fiedler vector and bisection | 3 |
| Cheeger inequality verification | 4 |
| Graph Fourier Transform and filtering | 5 |
| Spectral clustering (NJW) | 6 |
| Laplacian / Random Walk positional encodings | 7 |
| PageRank and preference ranking | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la
import scipy.linalg as sla

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
    mpl.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 120,
                         'font.size': 12, 'axes.titlesize': 13})
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {'primary': '#0077BB', 'secondary': '#EE7733',
          'tertiary': '#009988', 'error': '#CC3311',
          'neutral': '#555555', 'highlight': '#EE3377'}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

# Shared utilities
def build_laplacian(A):
    D = np.diag(A.sum(axis=1))
    return D - A

def build_normalized_laplacian(A):
    d = A.sum(axis=1)
    d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)
    D_inv_sqrt = np.diag(d_inv_sqrt)
    return D_inv_sqrt @ build_laplacian(A) @ D_inv_sqrt

def path_graph(n):
    A = np.zeros((n, n))
    for i in range(n-1): A[i,i+1] = A[i+1,i] = 1
    return A

def cycle_graph(n):
    A = path_graph(n); A[0,n-1] = A[n-1,0] = 1
    return A

def complete_graph(n):
    return np.ones((n,n)) - np.eye(n)

def star_graph(n):
    A = np.zeros((n,n)); A[0,1:] = A[1:,0] = 1
    return A

print('Setup complete.')

---

## Exercise 1 ★ — Laplacian Construction and Dirichlet Energy

**Graph $G$:** vertices $\{0,1,2,3,4\}$, edges $\{(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)\}$ (5-cycle + chord).

(a) Construct $A$, $D$, $L = D - A$ by hand (as NumPy arrays).

(b) Verify the identity $\mathbf{x}^\top L \mathbf{x} = \sum_{(i,j)\in E}(x_i - x_j)^2$ for $\mathbf{x} = [1, 2, 0, -1, 1.5]^\top$.

(c) Compute all eigenvalues of $L$. How many connected components does $G$ have?

(d) Compute $L_{\text{sym}} = D^{-1/2}LD^{-1/2}$. Verify its eigenvalues lie in $[0,2]$.

(e) **For AI:** Which eigenvector of $L$ is used for spectral bisection? What partition does it suggest?

In [ ]:
# Exercise 1: Your Solution
edges_1 = [(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)]
n1 = 5

# (a) Build matrices
A1 = None  # YOUR CODE HERE
D1 = None  # YOUR CODE HERE
L1 = None  # YOUR CODE HERE

# (b) Verify Dirichlet energy
x1 = np.array([1.0, 2.0, 0.0, -1.0, 1.5])
dirichlet_matrix_1 = None  # x1 @ L1 @ x1
dirichlet_edges_1 = None   # sum over edges

# (c) Eigenvalues
evals_1 = None  # YOUR CODE HERE

# (d) Normalized Laplacian
L_sym_1 = None  # YOUR CODE HERE
evals_sym_1 = None  # YOUR CODE HERE

# (e) Fiedler vector
u2_1 = None  # YOUR CODE HERE

print('A1:', A1)
print('evals_1:', evals_1)
print('u2_1:', u2_1)

In [ ]:
# Exercise 1: Solution
edges_1 = [(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)]
n1 = 5

# (a)
A1 = np.zeros((n1, n1))
for i, j in edges_1:
    A1[i,j] = A1[j,i] = 1
D1 = np.diag(A1.sum(axis=1))
L1 = D1 - A1

# (b)
x1 = np.array([1.0, 2.0, 0.0, -1.0, 1.5])
dirichlet_matrix_1 = x1 @ L1 @ x1
dirichlet_edges_1 = sum((x1[i] - x1[j])**2 for i, j in edges_1)

# (c)
evals_1 = np.sort(sla.eigvalsh(L1))
n_components_1 = int(np.sum(np.abs(evals_1) < 1e-8))

# (d)
L_sym_1 = build_normalized_laplacian(A1)
evals_sym_1 = np.sort(sla.eigvalsh(L_sym_1))

# (e) Fiedler vector = eigenvector for lambda_2
_, evecs_1 = sla.eigh(L1)
u2_1 = evecs_1[:, 1]  # second smallest eigenvector
S_1 = [i for i in range(n1) if u2_1[i] >= 0]
Sbar_1 = [i for i in range(n1) if u2_1[i] < 0]

header('Exercise 1: Laplacian Construction and Dirichlet Energy')
print(f'Adjacency matrix A:\n{A1.astype(int)}')
print(f'Eigenvalues of L: {np.round(evals_1, 4)}')
check_true('(c) G is connected (one zero eigenvalue)', n_components_1 == 1)
check_close('(b) Dirichlet energy: matrix form', dirichlet_matrix_1, dirichlet_edges_1)
check_true('(d) L_sym eigenvalues in [0, 2]',
           np.all(evals_sym_1 >= -1e-8) and np.all(evals_sym_1 <= 2+1e-8))
print(f'(e) Spectral bisection: S={S_1}, Sbar={Sbar_1}')
print(f'    Fiedler vector u_2 = {np.round(u2_1, 4)}')
print('\nTakeaway: The Fiedler vector assigns each node a real value; '
      'sign determines the bisection. Low lambda_2 -> bottleneck cut.')

---

## Exercise 2 ★ — Spectrum of Special Graphs

(a) Derive the eigenvalues of $L$ for the cycle graph $C_6$. Use the formula $\lambda_k = 2 - 2\cos(2\pi(k-1)/n)$.

(b) Compute $\lambda_2(C_6)$ and compare with $\lambda_2(P_6) = 2 - 2\cos(\pi/6)$. Which is more connected?

(c) Prove (numerically) that for $K_n$: all non-zero eigenvalues equal $n$, and $\lambda_2(K_n) = n$.

(d) For the star $S_n$ (hub + $n-1$ leaves), verify $\lambda_2 = 1$ for $n = 4, 6, 8, 10$. Explain why it is independent of $n$.

In [ ]:
# Exercise 2: Your Solution

# (a) Cycle C_6: analytic eigenvalues
n2 = 6
lambda_C6_analytic = None  # YOUR CODE HERE: array of 6 eigenvalues
lambda_C6_numeric  = None  # YOUR CODE HERE: compute from L

# (b) Compare lambda_2(C_6) vs lambda_2(P_6)
lam2_C6 = None  # YOUR CODE HERE
lam2_P6 = None  # YOUR CODE HERE

# (c) K_n: all non-zero eigenvalues = n
lam_Kn = None  # YOUR CODE HERE: eigenvalues of L for K_5

# (d) Star graph: lambda_2 independent of n
lam2_stars = {n: None for n in [4, 6, 8, 10]}  # YOUR CODE HERE

print('lambda_C6_analytic:', lambda_C6_analytic)
print('lam2_C6:', lam2_C6, 'vs lam2_P6:', lam2_P6)
print('lam2_stars:', lam2_stars)

In [ ]:
# Exercise 2: Solution

n2 = 6

# (a)
lambda_C6_analytic = np.array([2 - 2*np.cos(2*np.pi*k/n2) for k in range(n2)])
lambda_C6_analytic.sort()
lambda_C6_numeric = np.sort(sla.eigvalsh(build_laplacian(cycle_graph(n2))))

# (b)
lam2_C6 = lambda_C6_analytic[1]
lam2_P6 = 2 - 2*np.cos(np.pi/n2)

# (c)
n_K = 5
lam_Kn = np.sort(sla.eigvalsh(build_laplacian(complete_graph(n_K))))

# (d)
lam2_stars = {}
for n_s in [4, 6, 8, 10]:
    lam2_stars[n_s] = np.sort(sla.eigvalsh(build_laplacian(star_graph(n_s))))[1]

header('Exercise 2: Spectrum of Special Graphs')
check_close('(a) C_6 analytic eigenvalues match numeric',
            lambda_C6_analytic, lambda_C6_numeric, tol=1e-8)
print(f'(b) lambda_2(C_6) = {lam2_C6:.4f}, lambda_2(P_6) = {lam2_P6:.4f}')
check_true('(b) Cycle is more connected than path (larger lambda_2)', lam2_C6 > lam2_P6)
check_true('(c) K_5: all non-zero eigenvalues = 5',
           np.allclose(lam_Kn[1:], n_K, atol=1e-8))
print(f'(d) Star lambda_2 for different n: {lam2_stars}')
check_true('(d) Star lambda_2 = 1 for all n',
           all(np.isclose(v, 1.0, atol=1e-8) for v in lam2_stars.values()))
print('\nTakeaway: lambda_2 encodes connectivity. Star\'s lambda_2=1 regardless of size '
      'because removing the hub disconnects all leaves simultaneously.')

---

## Exercise 3 ★ — Fiedler Vector and Graph Bisection

**Barbell graph:** two copies of $K_5$, nodes $\{0\ldots4\}$ and $\{5\ldots9\}$, connected by a single bridge edge $(4, 5)$.

(a) Build the adjacency matrix $A$ and compute $\lambda_2$. Is the graph well-connected or a bottleneck?

(b) Compute the Fiedler vector $\mathbf{u}_2$. Report its values on each clique.

(c) Perform spectral bisection (partition by sign of $\mathbf{u}_2$). How many edges cross the cut?

(d) Compute the conductance $h(S, \bar{S}) = |E(S,\bar{S})| / \min(\text{vol}(S), \text{vol}(\bar{S}))$ for the spectral cut. Compare with the Cheeger bound $\sqrt{2\lambda_2}$.

(e) **For AI:** If this graph represented a distributed training cluster (nodes = GPUs, edges = high-bandwidth links), what does $\lambda_2$ tell you about all-reduce communication?

In [ ]:
# Exercise 3: Your Solution

def barbell_graph(n1=5, n2=5):
    # YOUR CODE HERE
    pass

A3 = barbell_graph(5, 5)

# (a)
lam2_3 = None  # YOUR CODE HERE

# (b)
u2_3 = None  # YOUR CODE HERE

# (c)
S_3, Sbar_3 = None, None  # YOUR CODE HERE
cut_edges_3 = None  # YOUR CODE HERE

# (d)
conductance_3 = None  # YOUR CODE HERE
cheeger_bound_3 = None  # YOUR CODE HERE

print('lam2:', lam2_3)
print('cut_edges:', cut_edges_3)
print('conductance:', conductance_3)

In [ ]:
# Exercise 3: Solution

def barbell_graph(n1=5, n2=5):
    n = n1 + n2
    A = np.zeros((n, n))
    for i in range(n1):
        for j in range(i+1, n1):
            A[i,j] = A[j,i] = 1
    for i in range(n1, n):
        for j in range(i+1, n):
            A[i,j] = A[j,i] = 1
    A[n1-1, n1] = A[n1, n1-1] = 1
    return A

A3 = barbell_graph(5, 5)
L3 = build_laplacian(A3)
evals3, evecs3 = sla.eigh(L3)

# (a)
lam2_3 = evals3[1]

# (b)
u2_3 = evecs3[:, 1]

# (c)
S_3 = [i for i in range(10) if u2_3[i] >= 0]
Sbar_3 = [i for i in range(10) if u2_3[i] < 0]
cut_edges_3 = sum(1 for i in S_3 for j in Sbar_3 if A3[i,j] > 0)

# (d)
d3 = A3.sum(axis=1)
vol_S = sum(d3[i] for i in S_3)
vol_Sbar = sum(d3[i] for i in Sbar_3)
L_sym3 = build_normalized_laplacian(A3)
lam2_sym3 = np.sort(sla.eigvalsh(L_sym3))[1]
conductance_3 = cut_edges_3 / min(vol_S, vol_Sbar)
cheeger_bound_3 = (2 * lam2_sym3) ** 0.5

header('Exercise 3: Fiedler Vector and Graph Bisection')
print(f'(a) lambda_2 = {lam2_3:.6f} (near 0 -> bottleneck graph)')
print(f'(b) Fiedler vector values:')
print(f'    Clique 1 (nodes 0-4): {np.round(u2_3[:5], 4)}')
print(f'    Clique 2 (nodes 5-9): {np.round(u2_3[5:], 4)}')
check_true('(c) Spectral bisection finds the 1-edge bridge cut', cut_edges_3 == 1)
check_true('(d) Conductance satisfies Cheeger upper bound',
           conductance_3 <= cheeger_bound_3 + 1e-6)
print(f'    Conductance h = {conductance_3:.4f}, Cheeger bound sqrt(2*lam2) = {cheeger_bound_3:.4f}')
print('\nTakeaway: Fiedler vector precisely identifies the bridge. '
      'For distributed training: small lambda_2 means slow all-reduce '
      '(communication bottleneck across the bridge).')

---

## Exercise 4 ★★ — Cheeger Inequality Verification

For the path graph $P_{10}$:

(a) Compute $\lambda_2(L_{\text{sym}})$ analytically and numerically.

(b) Find the optimal Cheeger cut by evaluating all $n-1$ possible balanced cuts of the path. Compute $h(G)$.

(c) Verify $\lambda_2/2 \leq h(G) \leq \sqrt{2\lambda_2}$ and compute how tight each bound is.

(d) Implement the Fiedler sweep algorithm: sort nodes by $\mathbf{u}_2$ value, evaluate conductance for each prefix, return the best cut.

(e) **For AI:** If the path were a chain of Transformer layers (layer $i$ passes activations to layer $i+1$), what does the Cheeger constant say about information bottlenecks in the network?

In [ ]:
# Exercise 4: Your Solution
n4 = 10
A4 = path_graph(n4)
L_sym4 = build_normalized_laplacian(A4)

# (a) Analytic and numeric lambda_2
lam2_analytic_4 = None  # 2 - 2*cos(pi/n4) ... but which formula for L_sym?
lam2_numeric_4 = None   # YOUR CODE HERE

# (b) Cheeger constant: brute-force over all n-1 path cuts
def path_cheeger(n):
    # YOUR CODE HERE: for path, cut at position k means S={0..k-1}, Sbar={k..n-1}
    pass

h4, best_cut_4 = None, None  # path_cheeger(n4)

# (c) Verify Cheeger inequality
lower_4 = None  # lam2/2
upper_4 = None  # sqrt(2*lam2)

# (d) Fiedler sweep
def fiedler_sweep(A):
    # YOUR CODE HERE
    pass

h_sweep_4, S_sweep_4 = None, None  # fiedler_sweep(A4)

print('lam2_numeric:', lam2_numeric_4)
print('h(G):', h4, 'best cut at:', best_cut_4)

In [ ]:
# Exercise 4: Solution
n4 = 10
A4 = path_graph(n4)
L_sym4 = build_normalized_laplacian(A4)

# (a) Note: L_sym eigenvalues != L eigenvalues for irregular graphs
# For path: L_sym eigenvalues are approximately 2*(1 - cos(k*pi/(n)))
lam2_numeric_4 = np.sort(sla.eigvalsh(L_sym4))[1]
# Analytic: L_sym of path ~ 1 - cos(pi/n) (leading term)
lam2_analytic_4 = 1 - np.cos(np.pi / n4)  # approx for L_sym of path

# (b) Path: cut at position k -> S = {0..k-1}, Sbar = {k..n-1}
def path_cheeger(n):
    A = path_graph(n)
    d = A.sum(axis=1)
    vol_total = d.sum()
    h_min = float('inf')
    best_k = -1
    for k in range(1, n):
        # cut edges: only edge (k-1, k)
        cut = 1
        vol_S = d[:k].sum()
        vol_Sbar = d[k:].sum()
        h = cut / min(vol_S, vol_Sbar)
        if h < h_min:
            h_min = h
            best_k = k
    return h_min, best_k

h4, best_cut_4 = path_cheeger(n4)

# (c)
lower_4 = lam2_numeric_4 / 2
upper_4 = (2 * lam2_numeric_4) ** 0.5

# (d) Fiedler sweep
def fiedler_sweep(A):
    L_sym = build_normalized_laplacian(A)
    evals, evecs = sla.eigh(L_sym)
    u2 = evecs[:, 1]
    n = A.shape[0]
    d = A.sum(axis=1)
    order = np.argsort(u2)
    h_min = float('inf')
    best_S = None
    for k in range(1, n):
        S = list(order[:k])
        Sbar = list(order[k:])
        cut = sum(A[i,j] for i in S for j in Sbar)
        vol_S = sum(d[i] for i in S)
        vol_Sbar = sum(d[i] for i in Sbar)
        if vol_S > 0 and vol_Sbar > 0:
            h = cut / min(vol_S, vol_Sbar)
            if h < h_min:
                h_min = h
                best_S = S
    return h_min, best_S

h_sweep_4, S_sweep_4 = fiedler_sweep(A4)

header('Exercise 4: Cheeger Inequality Verification')
print(f'(a) lambda_2(L_sym, P_10) = {lam2_numeric_4:.6f}')
print(f'(b) h(P_10) = {h4:.6f}, optimal cut at position {best_cut_4}')
print(f'(c) Cheeger bounds: [{lower_4:.6f}, {upper_4:.6f}]')
check_true('(c) lower bound: lambda_2/2 <= h(G)', lower_4 <= h4 + 1e-8)
check_true('(c) upper bound: h(G) <= sqrt(2*lambda_2)', h4 <= upper_4 + 1e-8)
print(f'    Tightness: lower={h4/lower_4:.2f}x, upper={h4/upper_4:.2f}x')
check_true('(d) Fiedler sweep finds cut with h <= sqrt(2*lambda_2)',
           h_sweep_4 <= upper_4 + 1e-8)
print(f'    Fiedler sweep: h = {h_sweep_4:.6f}')
print('\nTakeaway: For path graphs, the Cheeger constant is ~2/n (bottleneck at center). '
      'In a chain of Transformer layers, information must pass through each '
      'intermediate layer — no shortcuts.')

---

## Exercise 5 ★★ — Graph Fourier Transform and Spectral Filtering

Build a community graph: two cliques of 6 nodes each, connected by 2 bridge edges. Define a community signal $x_i = +1$ for community 1, $x_i = -1$ for community 2.

(a) Compute the GFT $\hat{\mathbf{x}} = U^\top \mathbf{x}$. Verify Parseval's theorem $\lVert \hat{\mathbf{x}} \rVert = \lVert \mathbf{x} \rVert$.

(b) Show that the community signal is concentrated in low frequencies: compute the fraction of energy in the first 4 frequency components.

(c) Add Gaussian noise: $\mathbf{x}' = \mathbf{x} + 0.5 \boldsymbol{\eta}$, $\boldsymbol{\eta} \sim \mathcal{N}(0, I)$. Apply a low-pass filter $g(\lambda) = \mathbf{1}[\lambda \leq \lambda_5]$ (keep first 5 frequencies). Compute the fraction of nodes correctly classified after filtering.

(d) Compare Dirichlet energies: $\mathbf{x}^\top L \mathbf{x}$ for $\mathbf{x}$, $\mathbf{x}'$, and the filtered output.

(e) **For AI:** Connect low-pass graph filtering to label propagation in semi-supervised GNNs. What does the filter $g(\lambda)$ correspond to in the GCN layer?

In [ ]:
# Exercise 5: Your Solution

def community_graph(n_per=6):
    # YOUR CODE HERE: two cliques of n_per nodes, connected by 2 bridge edges
    pass

A5, n_per5 = community_graph(6), 6
n5 = 12
x_comm5 = np.array([1.0]*n_per5 + [-1.0]*n_per5)

# (a) GFT and Parseval
L5 = build_laplacian(A5)
evals5, U5 = None, None  # YOUR CODE HERE
x_hat5 = None             # YOUR CODE HERE

# (b) Energy in low frequencies
energy_low_frac5 = None  # YOUR CODE HERE

# (c) Noisy signal + low-pass filter
np.random.seed(42)
x_noisy5 = x_comm5 + 0.5 * np.random.randn(n5)
x_filtered5 = None  # YOUR CODE HERE
accuracy5 = None    # YOUR CODE HERE

# (d) Dirichlet energies
E_clean5 = None   # YOUR CODE HERE
E_noisy5 = None   # YOUR CODE HERE
E_filter5 = None  # YOUR CODE HERE

print('energy_low_frac:', energy_low_frac5)
print('accuracy after filtering:', accuracy5)

In [ ]:
# Exercise 5: Solution

def community_graph(n_per=6):
    n = 2 * n_per
    A = np.zeros((n, n))
    for i in range(n_per):
        for j in range(i+1, n_per):
            A[i,j] = A[j,i] = 1
            A[n_per+i, n_per+j] = A[n_per+j, n_per+i] = 1
    A[n_per-1, n_per] = A[n_per, n_per-1] = 1
    A[n_per-2, n_per+1] = A[n_per+1, n_per-2] = 1
    return A

n_per5 = 6
A5 = community_graph(n_per5)
n5 = 12
x_comm5 = np.array([1.0]*n_per5 + [-1.0]*n_per5)

# (a)
L5 = build_laplacian(A5)
evals5, U5 = sla.eigh(L5)
x_hat5 = U5.T @ x_comm5

# (b)
total_energy5 = np.sum(x_hat5**2)
low_energy5 = np.sum(x_hat5[:4]**2)
energy_low_frac5 = low_energy5 / total_energy5

# (c) Low-pass filter: keep first 5 frequencies
np.random.seed(42)
x_noisy5 = x_comm5 + 0.5 * np.random.randn(n5)
x_hat_noisy5 = U5.T @ x_noisy5
x_hat_filtered5 = x_hat_noisy5.copy()
x_hat_filtered5[5:] = 0  # zero out high-frequency components
x_filtered5 = U5 @ x_hat_filtered5
accuracy5 = np.mean((x_filtered5 > 0) == (x_comm5 > 0))

# (d)
E_clean5 = x_comm5 @ L5 @ x_comm5
E_noisy5 = x_noisy5 @ L5 @ x_noisy5
E_filter5 = x_filtered5 @ L5 @ x_filtered5

header('Exercise 5: Graph Fourier Transform and Spectral Filtering')
check_close('(a) Parseval: ||x_hat|| = ||x||',
            np.linalg.norm(x_hat5), np.linalg.norm(x_comm5))
print(f'(b) Energy in first 4 frequencies: {energy_low_frac5:.1%} of total')
check_true('(b) Community signal concentrated in low frequencies (>60%)',
           energy_low_frac5 > 0.6)
print(f'(c) Classification accuracy after low-pass filter: {accuracy5:.1%}')
check_true('(c) Filtering improves classification (accuracy > 0.8)', accuracy5 > 0.8)
print(f'(d) Dirichlet energy: clean={E_clean5:.2f}, noisy={E_noisy5:.2f}, '
      f'filtered={E_filter5:.2f}')
check_true('(d) Filtering reduces Dirichlet energy toward clean signal', E_filter5 < E_noisy5)
print('\nTakeaway: Community signals live in low-frequency subspace. '
      'GCN acts as a low-pass filter: g(lambda) = 1-lambda/2 attenuates high frequencies, '
      'which is why it smooths node features toward neighborhood averages.')

---

## Exercise 6 ★★ — Spectral Clustering (NJW Algorithm)

Generate a Stochastic Block Model graph with **3 communities** of 30 nodes each, $p_{\text{in}} = 0.30$, $p_{\text{out}} = 0.02$.

(a) Plot the first 8 eigenvalues of $L_{\text{sym}}$. Where is the largest eigengap? Does it match the true $k=3$?

(b) Implement NJW spectral clustering: compute $k=3$ smallest eigenvectors of $L_{\text{sym}}$, row-normalize, apply k-means. Report the Adjusted Rand Index.

(c) Repeat with $p_{\text{out}} = 0.15$ (near phase transition). How does the eigengap and ARI change?

(d) Compare spectral clustering vs k-means applied directly to node features (assign feature $x_i = $ community index, plus Gaussian noise). Which method is more robust to noise?

(e) **For AI:** If a knowledge graph had community structure like this SBM, how would you use spectral clustering to create topic-specific sub-graphs for retrieval-augmented generation?

In [ ]:
# Exercise 6: Your Solution
try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import adjusted_rand_score
    HAS_SK = True
except ImportError:
    HAS_SK = False

def sbm(n_blocks, n_per, p_in, p_out, seed=42):
    # YOUR CODE HERE
    pass

def spectral_clustering_njw(A, k):
    # YOUR CODE HERE
    pass

A6, y6 = sbm(3, 30, 0.30, 0.02)
evals6 = None  # eigenvalues of L_sym
pred6  = None  # cluster predictions
ari6   = None  # ARI score

# Hard case
A6h, y6h = sbm(3, 30, 0.30, 0.15)
pred6h = None
ari6h  = None

print('evals6[:6]:', evals6[:6] if evals6 is not None else None)
print('ARI (easy):', ari6, 'ARI (hard):', ari6h)

In [ ]:
# Exercise 6: Solution
try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import adjusted_rand_score
    HAS_SK = True
except ImportError:
    HAS_SK = False

def sbm(n_blocks, n_per, p_in, p_out, seed=42):
    np.random.seed(seed)
    n = n_blocks * n_per
    A = np.zeros((n, n))
    labels = np.repeat(np.arange(n_blocks), n_per)
    for i in range(n):
        for j in range(i+1, n):
            p = p_in if labels[i] == labels[j] else p_out
            if np.random.rand() < p:
                A[i,j] = A[j,i] = 1
    return A, labels

def spectral_clustering_njw(A, k):
    Ls = build_normalized_laplacian(A)
    evals, evecs = sla.eigh(Ls)
    Uk = evecs[:, :k]
    norms = np.linalg.norm(Uk, axis=1, keepdims=True)
    norms = np.where(norms < 1e-10, 1.0, norms)
    Y = Uk / norms
    if HAS_SK:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        return km.fit_predict(Y), evals
    return np.random.randint(0, k, A.shape[0]), evals

A6, y6 = sbm(3, 30, 0.30, 0.02)
pred6, evals6 = spectral_clustering_njw(A6, k=3)
ari6 = adjusted_rand_score(y6, pred6) if HAS_SK else float('nan')

A6h, y6h = sbm(3, 30, 0.30, 0.15)
pred6h, evals6h = spectral_clustering_njw(A6h, k=3)
ari6h = adjusted_rand_score(y6h, pred6h) if HAS_SK else float('nan')

eigengap_easy = evals6[3] - evals6[2]
eigengap_hard = evals6h[3] - evals6h[2]

header('Exercise 6: Spectral Clustering')
print(f'Easy (p_out=0.02): eigenvalues[:6] = {np.round(evals6[:6], 4)}')
print(f'  Eigengap (lambda_4 - lambda_3) = {eigengap_easy:.4f}')
print(f'  ARI = {ari6:.4f}')
check_true('(b) NJW achieves ARI > 0.9 on easy SBM', ari6 > 0.9 or not HAS_SK)
print(f'\nHard (p_out=0.15): eigenvalues[:6] = {np.round(evals6h[:6], 4)}')
print(f'  Eigengap = {eigengap_hard:.4f} (smaller -> harder to cluster)')
print(f'  ARI = {ari6h:.4f}')
check_true('(c) Hard SBM has smaller eigengap than easy SBM',
           eigengap_hard < eigengap_easy)
print('\nTakeaway: Spectral clustering detects communities via eigengap. '
      'For a KG: spectral clusters = topic communities. '
      'Sub-graphs per community = topic-focused retrieval indices.')

---

## Exercise 7 ★★★ — Laplacian and Random Walk Positional Encodings

Positional encodings (PE) give each graph node a unique structural fingerprint.

(a) Implement LapPE: the top $k=4$ Laplacian eigenvectors as node features (skip the trivial $\mathbf{u}_1$).

(b) Implement RWPE: $\text{RWPE}(v)_j = (P^j)_{vv}$ for $j = 1, \ldots, k$, where $P = D^{-1}A$ is the random walk matrix.

(c) Verify that LapPE suffers from sign ambiguity: if you negate one eigenvector, the PE changes. Show RWPE does NOT have this issue.

(d) For each of three graph types (path $P_{10}$, cycle $C_{10}$, star $S_{10}$), compute both PEs. Are nodes in different positions distinguishable?

(e) **For AI:** Why do graph Transformers (GPS, Graphformer) use RWPE instead of raw eigenvector LapPE? What invariance does RWPE satisfy that LapPE does not?

In [ ]:
# Exercise 7: Your Solution

def lap_pe(A, k=4):
    # YOUR CODE HERE: return (n x k) matrix of eigenvectors (skip u_1)
    pass

def rwpe(A, k=4):
    # YOUR CODE HERE: return (n x k) matrix, column j = diag(P^{j+1})
    pass

# Test graphs
A7_path = path_graph(10)
A7_cycle = cycle_graph(10)
A7_star = star_graph(10)

# (a,b)
lpe7 = lap_pe(A7_path, k=4)
rpe7 = rwpe(A7_path, k=4)

# (c) Sign ambiguity
lpe7_flipped = None  # flip sign of first eigenvector column
sign_invariant = None  # True if RWPE is sign-invariant

print('LapPE shape:', lpe7.shape if lpe7 is not None else None)
print('RWPE shape:', rpe7.shape if rpe7 is not None else None)

In [ ]:
# Exercise 7: Solution

def lap_pe(A, k=4):
    L = build_laplacian(A)
    evals, evecs = sla.eigh(L)
    return evecs[:, 1:k+1]  # skip trivial eigenvector

def rwpe(A, k=4):
    n = A.shape[0]
    d = A.sum(axis=1)
    P = (A / np.where(d > 0, d, 1)[:, None])  # row-stochastic
    pe = np.zeros((n, k))
    Pk = np.eye(n)
    for j in range(k):
        Pk = Pk @ P
        pe[:, j] = np.diag(Pk)
    return pe

A7_path  = path_graph(10)
A7_cycle = cycle_graph(10)
A7_star  = star_graph(10)

# (a,b)
lpe7 = lap_pe(A7_path, k=4)
rpe7 = rwpe(A7_path, k=4)

# (c) Sign ambiguity of LapPE
lpe7_flipped = lpe7.copy()
lpe7_flipped[:, 0] *= -1  # flip first eigenvector
lpe_changed = not np.allclose(lpe7, lpe7_flipped)

# RWPE does not depend on eigenvector signs
sign_invariant = True  # RWPE computed from (P^j)_ii, no eigenvectors involved

# (d) Distinguish nodes by PE
for name, A_g in [('Path P_10', A7_path), ('Cycle C_10', A7_cycle),
                  ('Star S_10', A7_star)]:
    rpe_g = rwpe(A_g, k=4)
    n_unique = len(set(tuple(row) for row in np.round(rpe_g, 8)))
    print(f'{name}: {n_unique}/{A_g.shape[0]} nodes have unique RWPE '
          f'({"all distinct" if n_unique==A_g.shape[0] else "some identical"})')

header('Exercise 7: Laplacian and Random Walk Positional Encodings')
print(f'LapPE shape: {lpe7.shape}, RWPE shape: {rpe7.shape}')
check_true('(c) LapPE changes when eigenvector sign flipped', lpe_changed)
check_true('(c) RWPE is sign-invariant by construction', sign_invariant)
print('\nTakeaway: RWPE = (P^j)_vv = probability of returning to v after j steps. '
      'This quantity depends only on the graph structure, not eigenvector orientation. '
      'GPS uses RWPE to give Transformers position-awareness without sign ambiguity.')

---

## Exercise 8 ★★★ — PageRank and Preference-Based Ranking

(a) Build a directed citation graph (8 pages, given edges below). Implement PageRank via power iteration with $\alpha = 0.85$. Report ranks and verify they sum to 1.

(b) Compute the Google matrix $G = \alpha P + (1-\alpha)\mathbf{1}\mathbf{1}^\top/n$ and find its dominant eigenvector directly. Compare with power iteration.

(c) Add a dangling node (node 8, no outgoing edges). How does this affect $P$? Apply the standard fix (teleport from dangling nodes).

(d) Measure convergence: how many iterations does it take to reach $\lVert \boldsymbol{\pi}^{(t)} - \boldsymbol{\pi}^* \rVert < 10^{-8}$ for $\alpha \in \{0.50, 0.85, 0.99\}$? Connect to mixing time theory.

(e) **For AI (RLHF):** Given 5 model responses with pairwise preference comparisons (A beats B, B beats C, etc.), build a directed preference graph and run PageRank to produce a global ranking. Compare with simple win-count ranking.

In [ ]:
# Exercise 8: Your Solution

# (a) Citation graph
n8 = 8
edges_dir8 = [(0,1),(0,2),(1,3),(2,3),(2,4),(3,5),(4,5),(5,6),(6,0),(1,6),(2,6),(7,3)]
A8 = np.zeros((n8, n8))
for i, j in edges_dir8:
    A8[i,j] = 1

def pagerank_power(A_dir, alpha=0.85, max_iter=500, tol=1e-10):
    # YOUR CODE HERE
    pass

pi8, n_iters8 = None, None  # pagerank_power(A8)

# (b) Direct eigenvector computation
def build_google_matrix(A, alpha=0.85):
    # YOUR CODE HERE
    pass

pi8_eig = None  # YOUR CODE HERE

# (e) RLHF preference graph
responses = ['A', 'B', 'C', 'D', 'E']
preferences = [('A','B'),('A','C'),('B','D'),('C','D'),('D','E'),('B','E'),('A','E'),('C','E')]
A_pref = None  # YOUR CODE HERE: directed graph where edge (i,j) = response i beats j
pi_pref = None  # YOUR CODE HERE: PageRank of preference graph

print('PageRank:', pi8)
print('n_iters:', n_iters8)

In [ ]:
# Exercise 8: Solution

n8 = 8
edges_dir8 = [(0,1),(0,2),(1,3),(2,3),(2,4),(3,5),(4,5),(5,6),(6,0),(1,6),(2,6),(7,3)]
A8 = np.zeros((n8, n8))
for i, j in edges_dir8:
    A8[i, j] = 1

def build_google_matrix(A, alpha=0.85):
    n = A.shape[0]
    out_deg = A.sum(axis=1)
    out_deg_safe = np.where(out_deg == 0, 1, out_deg)
    P = (A.T / out_deg_safe).T
    # Handle dangling rows: redirect to uniform
    dangling = (out_deg == 0)
    P[dangling, :] = 1.0 / n
    return alpha * P + (1 - alpha) * np.ones((n, n)) / n

def pagerank_power(A, alpha=0.85, max_iter=500, tol=1e-10):
    G = build_google_matrix(A, alpha)
    n = A.shape[0]
    pi = np.ones(n) / n
    for t in range(max_iter):
        pi_new = pi @ G
        if np.linalg.norm(pi_new - pi) < tol:
            return pi_new, t+1
        pi = pi_new
    return pi, max_iter

# (a)
pi8, n_iters8 = pagerank_power(A8)

# (b)
G8 = build_google_matrix(A8)
evals_G8, evecs_G8 = np.linalg.eig(G8.T)
idx = np.argmax(np.real(evals_G8))
pi8_eig = np.real(evecs_G8[:, idx])
pi8_eig = np.abs(pi8_eig) / np.abs(pi8_eig).sum()

# (d) Convergence vs alpha
print('Convergence iterations by alpha:')
for alpha_val in [0.50, 0.85, 0.99]:
    pi_conv, iters = pagerank_power(A8, alpha=alpha_val, tol=1e-8)
    print(f'  alpha={alpha_val}: {iters} iterations')

# (e) RLHF preference ranking
resp_idx = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
preferences = [('A','B'),('A','C'),('B','D'),('C','D'),
               ('D','E'),('B','E'),('A','E'),('C','E')]
A_pref = np.zeros((5,5))
win_count = np.zeros(5)
for winner, loser in preferences:
    i, j = resp_idx[winner], resp_idx[loser]
    A_pref[i, j] = 1  # edge: winner -> loser
    win_count[i] += 1

pi_pref, _ = pagerank_power(A_pref, alpha=0.85)
pr_rank = np.argsort(-pi_pref)
wc_rank = np.argsort(-win_count)

header('Exercise 8: PageRank and Preference Ranking')
print('(a) PageRank scores:', dict(enumerate(np.round(pi8, 4))))
print(f'    Converged in {n_iters8} iterations')
check_close('(a) PageRank sums to 1', pi8.sum(), 1.0)
check_close('(b) Power iteration matches direct eigenvector', pi8, pi8_eig, tol=1e-4)
print(f'\n(e) RLHF preference ranking:')
print(f'    PageRank order: {[list(resp_idx.keys())[i] for i in pr_rank]}')
print(f'    Win-count order: {[list(resp_idx.keys())[i] for i in wc_rank]}')
print(f'    PageRank scores: {dict(zip(resp_idx.keys(), np.round(pi_pref, 4)))}')
print('\nTakeaway: PageRank provides a global ranking that accounts for who '
      'beats whom, not just win counts. A wins all -> highest PageRank. '
      'In RLHF, this is more robust than simple ELO or win-rate.')

---

## What to Review After Finishing

- [ ] **Exercise 1** — Can you prove L ≽ 0 from the quadratic form?
- [ ] **Exercise 2** — Do you know the closed-form spectra of $K_n$, $P_n$, $C_n$, $S_n$?
- [ ] **Exercise 3** — Can you explain why the Fiedler vector finds the minimum bisection?
- [ ] **Exercise 4** — Do you understand both directions of the Cheeger inequality proof?
- [ ] **Exercise 5** — Can you connect the GFT low-pass filter to label propagation?
- [ ] **Exercise 6** — Can you derive the NJW algorithm from the NCut relaxation?
- [ ] **Exercise 7** — Can you explain sign ambiguity in LapPE and why RWPE avoids it?
- [ ] **Exercise 8** — Can you explain convergence rate of PageRank in terms of $\alpha$?

## References

1. Fiedler, M. (1973). Algebraic connectivity of graphs. *Czechoslovak Math. J.*
2. Shi, J. & Malik, J. (2000). Normalized cuts and image segmentation. *IEEE TPAMI*.
3. Ng, A., Jordan, M. & Weiss, Y. (2002). On spectral clustering. *NeurIPS*.
4. Kipf, T. & Welling, M. (2017). Semi-supervised classification with GCNs. *ICLR*.
5. Defferrard, M. et al. (2016). Convolutional neural networks on graphs (ChebNet). *NeurIPS*.
6. Dwivedi, V. et al. (2022). Graph neural networks with learnable structural and positional representations. *ICLR*.
7. Rampasek, L. et al. (2022). Recipe for a General, Powerful, Scalable Graph Transformer (GPS). *NeurIPS*.
8. Spielman, D. (2019). *Spectral Graph Theory* (lecture notes, Yale).